# Exercise 1: Data Handling

#### Read data and handle missing values

In [ ]:
import pandas as pd
import numpy as np

#### Display summary statistics, and state the number of samples and features in the dataset.

In [ ]:
data = pd.read_excel("Tensile_Data.xlsx")

print("Shape: ", data.shape)

data.describe()

#### Check if there are any missing values. If yes, remove the objects.

In [ ]:
data_drop_rows = data.copy()

if data_drop_rows.isnull().values.any():
    rows_dropped = data_drop_rows.shape[0] - data_drop_rows.dropna().shape[0]

    print("Missing values found")
    print("Rows dropped:", rows_dropped)

    data_drop_rows = data_drop_rows.dropna()

In [ ]:
print("\nShape AFTER dropping rows: ", data_drop_rows.shape, "\n\n")

data_drop_rows.describe()

#### Remove the columns with missing values from original dataset

In [ ]:
data_drop_cols = data.copy()

data_drop_cols = data_drop_cols.dropna(axis=1)

print("Cols dropped: ", data_drop_cols.shape)

In [ ]:
data_drop_cols.describe()

## Exercise 2: Data Prediction

#### Read data and handle missing values

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

data_read = pd.read_csv("NewYork_1980_data.csv")

print("Shape:", data_read.shape)

#### Display summary statistics, and state the number of samples and features in the dataset.

In [ ]:
print("Shape: ", data_read.shape, "\n\n")
data_read.describe()

#### Show the first 5 rows using head() and last 10 rows using tail()

In [ ]:
data_read.head(5).iloc[:, -10:]

#### Check the counts for other columns. If there are any duplicate columns, remove them.

In [ ]:
if data_read.columns.duplicated().any():
  print("Yes, duplicate columns exist")
else:
  print("No duplicate columns exist")

#### Check if there are any missing values. If yes, remove the rows with missing values

In [ ]:
print(data_read.isnull().sum(), "\n\n")

data_read_clean = data_read.copy()
data_read_clean = data_read_clean.dropna()

print("Shape after dropping rows:", data_read_clean.shape)

### Check Feature Distribution

#### Create a histogram plot of the numerical features using matplotlib

In [ ]:
numerical_cols = ['age', 'education-num', 'capital-gain',
                  'capital-loss', 'hours-per-week']

focus_cols = ['age', 'hours-per-week']

#### Plot the distributions of age and hours-per-week. Based on the plots, state the your observation about relationship between these two features for each object.

In [ ]:
data_read_clean[numerical_cols].hist(figsize=(15, 10), bins=15)
plt.suptitle("All Numerical Features")
plt.tight_layout()
plt.show()

data_read_clean[focus_cols].hist(figsize=(12, 5), bins=15)
plt.suptitle("Age vs Hours-Per-Week Distribution")
plt.tight_layout()
plt.show()

### Encoding Categorical Features

#### Use the Ordinal Encoder in scikit-learn to encode all columns with text data, except for the column income.

In [ ]:
categorical_cols = ['workclass', 'education', 'marital-status',
                    'occupation', 'relationship', 'race',
                    'sex', 'native-country']

oe = OrdinalEncoder()
data_read_clean[categorical_cols] = oe.fit_transform(
                                    data_read_clean[categorical_cols])

#### Use the Label Encoder in scikit-learn to encode the feature income.

In [ ]:
le = LabelEncoder()
data_read_clean['income'] = le.fit_transform(data_read_clean['income'])

### Scaling Numerical Features

#### Create target labels named y consisting of the income column, and a data label named X including all remaining columns.

In [ ]:
print(data_read_clean[categorical_cols + ['income']].head())

#### Use standardization to scale all columns in the data X to have 0 mean and 1 standard deviation.

In [ ]:
y = data_read_clean['income']
X = data_read_clean.drop(columns=['income'])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Mean of each column (should be ~0):")
print(X_scaled.mean(axis=0).round(2))
print("\nStd of each column (should be ~1):")
print(X_scaled.std(axis=0).round(2))

#### Sampling the dataset: Generate a new dataset with 300 objects using any 2 types of sample techniques

In [ ]:
print(y.value_counts(normalize=True))

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

random_sample = X_scaled_df.sample(n=300, random_state=42)

print("Random sample shape:", random_sample.shape)
print("\n\nClass proportions:")
print(y[random_sample.index].value_counts(normalize=True))

In [ ]:
X_strat, _, y_strat, _ = train_test_split(
    X_scaled_df, y,
    test_size=len(X_scaled_df) - 300,
    stratify=y,
    random_state=42
)

print("Stratified sample shape:", X_strat.shape)
print("\n\nClass proportions:")
print(y_strat.value_counts(normalize=True))

# Exercise 3 Data Handling

#### Read data

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

df = pd.read_excel("Amazon.xlsx", na_values=['?'])

#### Display summary statistics, and state the number of samples and features in the dataset. How many rows and columns are in the dataframe?

In [ ]:
print("Shape:", df.shape, "\n\n")
df.describe()

#### Display the first 10 rows of the table, using the head() function.

In [ ]:
df.head(10)

#### Display the last 10 rows of the table, using the tail() function.

In [ ]:
df.tail(10)

#### Display the column names (using the .columns attribute).

In [ ]:
df.columns

### Handle missing values

#### Check if there are any missing values, use any strategy to solve it. Hint: if 'Adj close' column missed value, use 'Close' column's value to replace, vice verse. For some features replace the missing values with the 5 close days' average value of the column. or direct to remove the objects

In [ ]:
data_copy = df.copy()

print("Missing values:")
print(df.isnull().sum())

data_copy['Adj Close'] = data_copy['Adj Close'].fillna(data_copy['Close'])
data_copy['Close'] = data_copy['Close'].fillna(data_copy['Adj Close'])

cols_to_fill = ['Open', 'High', 'Low', 'Volume']

for col in cols_to_fill:
    data_copy[col] = data_copy[col].fillna(
        data_copy[col].rolling(window=5, min_periods=1).mean()
    )

print("\n\nMissing values after fixing:")
print(data_copy.isnull().sum())

In [ ]:
print("Shape: ", data_copy.shape, "\n\n")
data_copy.describe()

### Handle duplicated rows

#### Check if there are any duplicated records (if 'Date' column has the same value). If yes, solve the issue using any strategy. Hint: keep the record without missing values. If all the records are valide, keep the first one or last one.

In [ ]:
data_copy_nodups = data_copy.copy()

print("Duplicate dates before:")
print(data_copy_nodups.duplicated(subset=['Date']).sum())

data_copy_nodups = data_copy_nodups.drop_duplicates(subset=['Date'], keep='first')

print("\n\nDuplicate dates after:", data_copy_nodups.duplicated(subset=['Date']).sum())
print("\n\nShape after:", data_copy_nodups.shape)

### Handle outlier

#### Check if thre are any outliers by plotting the curves for each column. If the outlier exists, 1) remove the records; 2) use Z-score for 5 close day's average to replace the outlier value.

In [ ]:
cols_to_plot = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

fig, axes = plt.subplots(len(cols_to_plot), 1, figsize=(12, 20))

for i, col in enumerate(cols_to_plot):
    axes[i].plot(data_copy_nodups['Date'], data_copy_nodups[col])
    axes[i].set_title(col)
    axes[i].set_ylabel('Value')

plt.tight_layout()
plt.show()

In [ ]:
df_remove_outliers = data_copy_nodups.copy()

cols_to_check = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

z_scores = np.abs(stats.zscore(df_remove_outliers[cols_to_check]))

mask = (z_scores < 3).all(axis=1)
df_remove_outliers = df_remove_outliers[mask]

print("Shape before:", data_copy_nodups.shape, "\n\n")
print("Shape after removing outliers:", df_remove_outliers.shape, "\n\n")
print("Outlier rows removed:", data_copy_nodups.shape[0] - df_remove_outliers.shape[0])

In [ ]:
df_replace_outliers = data_copy_nodups.copy()

for col in cols_to_check:
    z = np.abs(stats.zscore(df_replace_outliers[col]))
    outlier_mask = z > 3
    
    rolling_mean = df_replace_outliers[col].rolling(
                   window=5, min_periods=1).mean()
    df_replace_outliers.loc[outlier_mask, col] = rolling_mean[outlier_mask]

print("Outliers replaced in df_replace_outliers")
print("\n\nShape:", df_replace_outliers.shape)

#### Sample 1000 points and Plot the stock's trend using the column [high]'s value

In [ ]:
df_sampled = df_replace_outliers.sample(n=1000, random_state=42)

df_sampled = df_sampled.sort_values('Date')

plt.figure(figsize=(14, 5))
plt.plot(df_sampled['Date'], df_sampled['High'])
plt.title("Amazon Stock - High Price Trend (1000 sampled points)")
plt.xlabel("Date")
plt.ylabel("High Price ($)")
plt.tight_layout()
plt.show()